# Table ARI values (alternative to the overview plot)

Note: this takes computed ARI values from the "figure" script and does not compute the ARI values locally.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import corc.graph_metrics.neb
import corc.utils
import corc.our_datasets
import os
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

cache_path="../../cache"

In [4]:
# filename = os.path.join(cache_path, "metrics", "figure2_main.pkl")
# with open(filename, "rb") as f:
#     data = pickle.load(f)
# filename = os.path.join(cache_path, "metrics", "figure2_main-full.pkl")
filename = os.path.join(cache_path, "metrics", "figure2-full.pkl")
filename = os.path.join(cache_path, "metrics", "figure1-full.pkl")
with open(filename, "rb") as f:
    data = pickle.load(f)

In [5]:
data.keys()

dict_keys(['noisy_circles', 'noisy_moons', 'blobs', 'varied', 'aniso', 'clusterlab10'])

In [5]:
data["densired8"]["TMM-NEB"]
# 4 rows for: ARI, NMI, Folkwes-Mallows, and Variation of Information

[0.9154259542207419,
 0.8440202149011583,
 0.9134772967188428,
 0.8432534566657195,
 0.8441253272565572,
 0.8444988035981547,
 0.9136137551431639,
 0.8429034928364494,
 0.8439927963059909,
 0.9141366986510278]

In [8]:
import corc.our_algorithms
def style_col(col):
    ranked = col.rank(method='min', ascending=False)
    return ['background-color: lightgreen' if ranked.loc[i] == 1 else 
            'background-color: orange' if ranked.loc[i] == 2 else 
            'background-color: lightblue' if ranked.loc[i] == 3 else '' 
            for i in col.index]

def style_df(df):
    styled_df = df.style
    for col in df.columns:
        styled_df = styled_df.apply(style_col, subset=[col])
    return styled_df

def get_styled_table(data, operation=np.mean, core_algos=True):
    table = []
    for key in data.keys(): # datasets
        dict2 = data[key]
        row = []
        # print(dict2.keys())
        for key, value in dict2.items(): # algorithms
            if key not in corc.our_algorithms.CORE_SELECTOR and core_algos:
                continue
            # print(value)
            row.append(operation(value))
            # row.append(d[0])
        table.append(row)

    dataset_labels = [corc.our_datasets.dataset_displaynames[key] for key in data.keys()]
    algorithm_labels = list(data[list(data.keys())[0]].keys())
    if core_algos:
        algorithm_labels = [algo for algo in algorithm_labels if algo in corc.our_algorithms.CORE_SELECTOR]
    algorithm_labels = [corc.our_algorithms.ALG_DISPLAYNAMES[algo] if algo in corc.our_algorithms.ALG_DISPLAYNAMES else algo for algo in algorithm_labels]

    dataset_labels = [dataset.replace("\n", " ") for dataset in dataset_labels]
    algorithm_labels = [algo.replace("\n", " ") for algo in algorithm_labels]
    df = pd.DataFrame(table, columns=algorithm_labels, index=dataset_labels)
    df = df.transpose()
    # df
    df = df.map(lambda x: f"{x:.2f}")
    styled_df = style_df(df)
    return styled_df

get_styled_table(data, np.mean , core_algos=True)

,Noisy circles,Noisy moons,Gaussian blobs,Varied density,Anisotropic blobs,Clusterlab10
Agglomerative Clustering,-0.00,0.51,0.96,0.95,0.48,1.00
HDBSCAN,1.00,1.00,0.81,0.88,0.94,1.00
Gaussian Mixture,-0.00,0.40,0.06,0.01,0.47,0.25
Leiden,1.00,1.00,0.96,0.96,0.90,1.00
UniForCE,-0.00,0.18,0.55,0.38,0.35,0.00
SMMP,1.00,1.00,0.69,0.70,0.32,0.14
GWG-dip,0.35,1.00,0.68,0.84,0.63,0.54
g-NEB (ours),1.00,1.00,0.97,0.95,0.98,0.99
t-NEB (ours),1.00,1.00,0.98,0.58,0.99,1.00


In [15]:
data_copy = data.copy()

In [19]:
data_copy = pd.DataFrame(data_copy)

In [20]:
data_copy.columns

Index(['densired8', 'densired16', 'densired32', 'densired64',
       'densired_soft_8', 'densired_soft_16', 'densired_soft_32',
       'densired_soft_64', 'mnist8', 'mnist16', 'mnist32', 'mnist64'],
      dtype='object')

In [ ]:
for c in data_copy.columns:
    data_copy[c] = data_copy[c].apply(
        lambda x: '{np.mean(x)} \pm {np.std(x)}'
    )

In [19]:
get_styled_table(data, np.max)

,Densired 'circles' 8D,Densired 'circles' 16D,Densired 'circles' 32D,Densired 'circles' 64D,Densired 'Stud-t' 8D,Densired 'Stud-t' 16D,Densired 'Stud-t' 32D,Densired 'Stud-t' 64D,MNIST-Nd 8D,MNIST-Nd 16D,MNIST-Nd 32D,MNIST-Nd 64D
Agglomerative Clustering,0.68,0.66,0.59,0.75,0.56,0.87,0.90,0.64,0.80,0.68,0.62,0.49
HDBSCAN,0.00,0.00,0.44,0.00,0.01,0.00,0.00,0.00,0.03,0.06,0.07,0.07
Gaussian Mixture,0.79,0.77,0.69,0.91,0.74,0.68,0.49,0.42,0.89,0.74,0.73,0.62
Leiden,0.83,0.77,0.76,0.89,0.89,0.93,0.91,0.80,0.89,0.92,0.93,0.71
GWG-dip,0.98,1.00,1.00,0.98,0.09,0.72,0.25,0.50,0.89,0.85,0.49,0.53
UniForCE,0.32,0.24,0.47,0.14,0.25,0.34,0.45,0.78,0.13,0.13,0.05,0.03
SMMP,0.81,0.74,0.67,0.90,0.15,0.28,0.52,0.39,0.36,0.56,0.23,0.13
g-NEB (ours),1.00,1.00,1.00,0.99,0.77,0.72,0.74,0.34,0.77,0.62,0.56,0.40
t-NEB (ours),0.92,1.00,0.94,0.96,0.89,0.94,0.94,0.74,0.77,0.92,0.79,0.64


In [23]:
get_styled_table(data, lambda x: np.max(x)-np.min(x))

,Densired 'circles' 8D,Densired 'circles' 16D,Densired 'circles' 32D,Densired 'circles' 64D,Densired 'Stud-t' 8D,Densired 'Stud-t' 16D,Densired 'Stud-t' 32D,Densired 'Stud-t' 64D,MNIST-Nd 8D,MNIST-Nd 16D,MNIST-Nd 32D,MNIST-Nd 64D
Agglomerative Clustering,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
HDBSCAN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Gaussian Mixture,0.35,0.14,0.20,0.38,0.35,0.30,0.48,0.22,0.14,0.13,0.10,0.11
Leiden,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
GWG-dip,0.47,0.34,0.24,0.22,0.08,0.72,0.19,0.50,0.21,0.26,0.35,0.26
UniForCE,0.38,0.23,0.49,0.11,0.18,0.10,0.49,0.46,0.11,0.09,0.03,0.02
SMMP,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
g-NEB (ours),0.02,0.00,0.00,0.06,0.15,0.72,0.74,0.34,0.14,0.17,0.15,0.26
t-NEB (ours),0.07,0.09,0.22,0.19,0.14,0.19,0.31,0.19,0.12,0.14,0.05,0.21


In [9]:
get_styled_table(data, np.std)

,Noisy circles,Noisy moons,Gaussian blobs,Varied density,Anisotropic blobs,Clusterlab10
Agglomerative Clustering,0.00,0.00,0.00,0.00,0.00,0.00
HDBSCAN,0.00,0.00,0.00,0.00,0.00,0.00
Gaussian Mixture,0.00,0.00,0.00,0.00,0.00,0.00
Leiden,0.00,0.00,0.00,0.00,0.00,0.00
UniForCE,0.00,0.16,0.34,0.15,0.22,0.00
SMMP,0.00,0.00,0.00,0.00,0.00,0.00
GWG-dip,0.37,0.00,0.33,0.17,0.24,0.17
g-NEB (ours),0.00,0.00,0.00,0.00,0.01,0.01
t-NEB (ours),0.00,0.00,0.00,0.11,0.00,0.01


In [8]:
my_table = styled_df.to_latex(
    hrules=True, 
    label="tab:overview_hd", 
    caption="Overview of the performance of the algorithms on the datasets. The best performing algorithm is highlighted in green, the second best in orange, and the third best in light blue.",
    convert_css=True,
)
my_table = my_table.replace("lightgreen", "darkblue!60")
my_table = my_table.replace("orange", "darkblue!30")
my_table = my_table.replace("lightblue", "darkblue!10")
print(my_table)

\begin{table}
\caption{Overview of the performance of the algorithms on the datasets. The best performing algorithm is highlighted in green, the second best in darkblue!30, and the third best in light blue.}
\label{tab:overview_hd}
\begin{tabular}{lllllllllllll}
\toprule
 & Densired 'circles' 8D & Densired 'circles' 16D & Densired 'circles' 32D & Densired 'circles' 64D & Densired 'Stud-t' 8D & Densired 'Stud-t' 16D & Densired 'Stud-t' 32D & Densired 'Stud-t' 64D & MNIST-Nd 8D & MNIST-Nd 16D & MNIST-Nd 32D & MNIST-Nd 64D \\
\midrule
Agglomerative Clustering & 0.68 & 0.66 & 0.59 & 0.75 & 0.56 & {\cellcolor{darkblue!10}} 0.87 & {\cellcolor{darkblue!10}} 0.90 & 0.64 & 0.80 & 0.68 & 0.62 & 0.49 \\
HDBSCAN & 0.00 & 0.00 & 0.44 & 0.00 & 0.01 & 0.00 & 0.00 & 0.00 & 0.03 & 0.06 & 0.07 & 0.07 \\
Gaussian Mixture & 0.79 & 0.77 & 0.69 & 0.91 & 0.74 & 0.68 & 0.49 & 0.42 & {\cellcolor{darkblue!60}} 0.89 & 0.74 & {\cellcolor{darkblue!10}} 0.73 & {\cellcolor{darkblue!10}} 0.62 \\
Leiden & 0.83 & 0.77 